# Chapter 3 — Fitting a model · Notebook to complete

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch03_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to modify are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without raising an error. The solution is in `ch03_corrige.ipynb`, on the same companion page.

**Loss used**: the mean squared error (equation 3.2) on homes A to F, unless stated otherwise.

**Contents.** Data · The computations of lessons 3.1 to 3.6 · Exercises 3.1 to 3.6

*The cell of lesson 3.4 and the cell of exercise 3.4 f) run hundreds of thousands of steps: allow a few seconds for each.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book colors: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifier D0.1 is the one used in the book and in the CSV file.

In [ ]:
# Chapter 3 data, copied from the book's CSV file (do not modify)

# D0.1 — the six homes from the chapter opening, used for fitting
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)

# D3.1 — four new sales, used only for evaluation (lesson 3.6)
new_sales = pd.DataFrame({
    'sale':      ['V1', 'V2', 'V3', 'V4'],
    'floor_area': [38, 58, 76, 92],
    'price':  [132, 215, 254, 322],
})
xv = new_sales['floor_area'].to_numpy(dtype=float)
yv = new_sales['price'].to_numpy(dtype=float)

# Home G (chapter opening): worked example, never used for fitting
x_G, y_G = 62, 209

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `sale` | identifier of the new sale (V1 to V4) | — | `vente` |

## The computations of the lessons, in Python

These cells redo the computations of lessons 3.1 to 3.6.

In [ ]:
# Lesson 3.1 — two lines, two error measures (equations 3.1 and 3.2)
def mae(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for name, a, b in [("opening", 3, 30), ('colleague', 3.3, 14)]:
    print(f'{name:10s} MAE = {mae(a, b):.2f} k€   MSE = {mse(a, b):.2f} k€²   RMSE = {np.sqrt(mse(a, b)):.2f} k€')

In [ ]:
# Lesson 3.2 — the loss curve, b fixed at 30 k€ (equation 3.3)
slopes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(mse(a, 30), 2) for a in slopes], index=slopes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [mse(a, 30) for a in A], color=BLUE)
plt.scatter(slopes, [mse(a, 30) for a in slopes], color='black', zorder=3)
plt.xlabel('Slope a (k€ per m²)'); plt.ylabel('Loss L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 3.3 — secants, derivative (equation 3.4) and two updates (equation 3.5)
def derivative_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'secant h = {h}: {(mse(2.9 + h, 30) - mse(2.9, 30)) / h:.1f}')
print('exact derivative at 2.9:', round(derivative_a(2.9), 1))

a, eta = 2.5, 0.00006
for stage in range(3):
    print(f'step {stage}: a = {a}, L = {mse(a, 30):.2f}, L\'(a) = {derivative_a(a):.2f}')
    a = round(a - eta * derivative_a(a), 3)    # rounded to the thousandth, as in the book

In [ ]:
# Lesson 3.4 — the gradient (equation 3.6) and one step on both coefficients (equation 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('gradient at (2.5, 30):', g.round(2))
print('after one step:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'after {k} steps: a = {a:.4f}, b = {b:.4f}, L = {mse(a, b):.2f}')
a_ls, b_ls = np.polyfit(x, y, 1)
print(f'bottom of the valley: a = {a_ls:.3f}, b = {b_ls:.2f}, L = {mse(a_ls, b_ls):.2f}')
print(f'G, worked example: {a_ls * x_G + b_ls:.0f} k€ predicted for {y_G} k€ observed')

In [ ]:
# Lesson 3.5 — two descents starting from the same predictions (equation 3.8)
mu, sigma = x.mean(), x.std()          # std() divides by n, as in chapter 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descent_m2(eta, n_steps, a=2.5, b=30.0):
    for _ in range(n_steps):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descent_std(eta, n_steps, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # equation 2.7: same predictions at the start
    for _ in range(n_steps):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # equation 3.8: back to (a, b)

print('steps    | m², η = 0.00006           | standardized, η = 0.3')
a, b = 2.5, 30.0
done = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descent_m2(0.00006, k - done, a, b); done = k
    a_s, b_s = descent_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {mse(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {mse(a_s, b_s):8.2f}')

# Limit learning rates: beyond them, the descent diverges (about 0.00022 in m², 1 when standardized)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: loss after 400 steps = {mse(*descent_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'standardized, η = {eta}: loss after 400 steps = {mse(*descent_std(eta, 400)):.4g} k€²')

In [ ]:
# Lesson 3.6 — least-squares line (equation 3.9) and R² (equation 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_ls = cov / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_ls:.3f} k€/m², b = {b_ls:.2f} k€')

def r2_lesson(observed, predicted, reference):
    """R² = 1 − MSE of the rule / MSE of the constant prediction “reference”."""
    return 1 - np.mean((observed - predicted) ** 2) / np.mean((observed - reference) ** 2)

y_mean = y.mean()                      # mean price of A to F: the reference for R² on A to F
print(f'A to F: constant MSE = {np.mean((y - y_mean) ** 2):.2f}, line MSE = {mse(a_ls, b_ls):.2f}, '
      f'R² = {r2_lesson(y, a_ls * x + b_ls, y_mean):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# The three lines, the same measure, on A to F then on V1 to V4 (lesson 3.6)
lines = {'least squares': (a_ls, b_ls), 'opening': (3, 30), 'colleague': (3.3, 14)}
for name, (a, b) in lines.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{name:16s} A to F: MSE = {np.mean(e_af ** 2):6.2f}  MAE = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 to V4: MSE = {np.mean(e_v ** 2):6.2f}  MAE = {np.mean(np.abs(e_v)):5.2f}   deviations {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # if V4 had sold for 312 k€
for name in ('least squares', 'colleague'):
    a, b = lines[name]
    print(f'V4 at 312 k€: {name} MSE = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, worked example: {a_ls * x_G + b_ls:.2f} k€ predicted, deviation {y_G - (a_ls * x_G + b_ls):.2f} k€')

## Exercise 3.1 — Reading a graph: two measures, two minimums

*Data D0.1. Concepts: MAE and MSE (lesson 3.1), loss curve (lesson 3.2), median (lesson 1.3).* The slope is fixed at 3 k€/m². First answer questions a) to e) by reading the figure in the book.

### a) to e) Your readings

*Your answers:* …

### f) In the notebook

In [ ]:
# TO COMPLETE: MSE and MAE of the line 3x + b on A to F, for a given value of b
def mse_b(b):
    return None   # TO COMPLETE

def mae_b(b):
    return None   # TO COMPLETE

In [ ]:
# Check, then plot of the two curves
if mse_b(30) is None or mae_b(30) is None:
    print('Complete mse_b and mae_b.')
else:
    print('mse_b and mae_b:', 'correct' if np.isclose(mse_b(30), 344 / 6) and np.isclose(mae_b(30), 44 / 6) else 'check again')
    B = np.linspace(15, 45, 601)
    fig, (h, low) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [mse_b(b) for b in B], color=BLUE); h.set_ylabel('MSE (k€²)'); h.grid(alpha=0.4)
    low.plot(B, [mae_b(b) for b in B], color=ORANGE); low.set_ylabel('MAE (k€)'); low.grid(alpha=0.4)
    low.set_xlabel('Intercept b (k€)')
    plt.show()
    print('corners of the MAE curve (sorted r_i):', np.sort(y - 3 * x))

In [ ]:
# TO COMPLETE: median (equation 1.4) and mean of the r_i = y_i − 3 x_i
median_r = None
mean_r = None

In [ ]:
# Check
for name, yours, truth in [('median', median_r, 29.5), ('mean', mean_r, 182 / 6)]:
    print(f'{name} of the r:', 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again'))
if median_r is not None and mae_b(30) is not None:
    print(f'MAE at the median: {mae_b(median_r):.2f} k€ (on the plateau); '
          f'MSE at the mean: {mse_b(mean_r):.2f} k€² (minimum)')

*Your conclusion:* …

## Exercise 3.2 — Two steps of gradient descent

*Data D0.1. Concepts: mean squared error (lesson 3.1), derivative and update (lesson 3.3).* The slope is fixed at 3 k€/m². We are looking for the intercept $b$. First do questions a) to d) on paper.

### a) Prepare

In [ ]:
# TO COMPLETE: r_i = y_i − 3 x_i for the six homes (numpy array)
r = None

In [ ]:
# Check
if r is None:
    print('Complete the previous cell.')
else:
    print('r:', 'correct' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'check again')

### b) and c) On paper

Enter your results here: losses in k€², values of $b$ in k€.

In [ ]:
# TO COMPLETE: your results computed by hand
L_b0 = None       # loss at b0 = 20
derivative_b0 = None # L'(20)
b1 = None
L_b1 = None
b2 = None
L_b2 = None

In [ ]:
# Check (independent computation)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
expected_values = {'L_b0': (L_b0, _L(20)), 'derivative_b0': (derivative_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for name, (yours, truth) in expected_values.items():
    status = 'not yet completed' if yours is None else ('correct' if abs(yours - truth) < 0.01 else 'check again')
    print(f'{name}: {status}')

### d) Interpret

*Your answer:* …

### e) In the notebook

In [ ]:
# TO COMPLETE
def loss(b):
    """MSE of the line y = 3x + b on A to F (equation 3.2)."""
    return None   # TO COMPLETE

def derivative(b):
    """L'(b) = −(2/n) Σ e_i (equation 3.6)."""
    return None   # TO COMPLETE

<details><summary><b>Hint</b> (click if you get stuck)</summary>

First compute the array of deviations `e = y - (3 * x + b)`. `np.mean(e ** 2)` gives the loss and `np.sum(e)` the sum of the deviations.

</details>

In [ ]:
def descent(b0, eta, n_steps):
    """Returns the list of b values and the list of losses, from step 0 to step n_steps."""
    bs, losses = [b0], [loss(b0)]
    for _ in range(n_steps):
        bs.append(bs[-1] - eta * derivative(bs[-1]))   # equation 3.5
        losses.append(loss(bs[-1]))
    return bs, losses

if loss(20) is None or derivative(20) is None:
    print('Complete loss and derivative.')
else:
    bs, losses = descent(20, 0.3, 2)
    print('two steps:', np.round(bs, 3), np.round(losses, 2), '→',
          'correct' if np.allclose(bs, [20, 26.2, 28.68]) else 'check again')
    bs, losses = descent(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(losses, marker='o', color=BLUE)
    plt.xlabel('Step'); plt.ylabel('Loss (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b after 20 steps:', round(bs[-1], 3), '  mean of the r:', round(np.mean(y - 3 * x), 3))

### f) Predict, then check: $\eta = 1.2$

*Your prediction, before running:* …

In [ ]:
if loss(20) is not None and derivative(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, losses = descent(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLUE)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Step'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Worked example on G

In [ ]:
# TO COMPLETE: predicted price for G with slope 3 and b2, then signed deviation (k€)
price_G = None
gap_G = None

In [ ]:
# Check
if price_G is None or gap_G is None:
    print('Complete the previous cell.')
else:
    print('G:', 'correct' if np.isclose(price_G, 214.68) and np.isclose(gap_G, -5.68) else 'check again')

*Your comment on G:* …

## Exercise 3.3 — What does "99%" mean?

*Data D0.1 and D3.1. Concepts: coefficient of determination, fitting and evaluation (lesson 3.6).* First do questions a) to f) on paper.

### a) to f) On paper

*Your answers:* …

### g) In the notebook

In [ ]:
# TO COMPLETE: MSE of predictions, on the sales passed to it (equation 3.2)
def sales_mse(observed, predicted):
    """observed and predicted: numpy arrays of the same length."""
    return None   # TO COMPLETE

In [ ]:
# Check: the ranking of lesson 3.6, then R² on A to F (reference: mean price of A to F)
if sales_mse(y, y) is None:
    print('Complete the function sales_mse.')
else:
    a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_ls = y.mean() - a_ls * x.mean()
    expected_values = {'least squares': ((a_ls, b_ls), 45.36, 93.29), 'opening': ((3, 30), 57.33, 134.25),
                'colleague': ((3.3, 14), 65.11, 70.73)}
    for name, ((a, b), att_af, att_v) in expected_values.items():
        e1, e2 = sales_mse(y, a * x + b), sales_mse(yv, a * xv + b)
        print(f'{name:16s} A to F: {e1:6.2f}   V1 to V4: {e2:6.2f} →',
              'correct' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'check again')
    r2_af = 1 - sales_mse(y, a_ls * x + b_ls) / sales_mse(y, np.full(6, y.mean()))
    print(f'R² on A to F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'correct' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'check again')

## Exercise 3.4 — A descent on standardized areas

*Data D0.1. Concepts: gradient (lesson 3.4), standardization and conversion back (lesson 3.5), least-squares line (lesson 3.6).*

### a) On paper

*Your computation:* …

### b) Mean, standard deviation and standardized scores

In [ ]:
# TO COMPLETE: mean and standard deviation of the floor areas (division by n), then the six z-scores
mu = None
sigma = None
z = None

In [ ]:
# Check
if z is None:
    print('Complete the previous cell.')
else:
    print('z-scores:', 'correct' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'check again')

### c) The gradient in the standardized form

In [ ]:
# TO COMPLETE: the two partial derivatives (lesson 3.5), returned in an array [dL/dw, dL/dc]
def standardized_gradient(w, c):
    return None   # TO COMPLETE

In [ ]:
# Check at the starting point (same predictions as (2.5, 30))
if z is None or standardized_gradient(1, 1) is None:
    print('Complete cells b) and c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = standardized_gradient(w0, c0)
    print(f'start: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'correct' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'check again')

### d) Thirty steps, then back to the original units

In [ ]:
# TO COMPLETE: 30 updates with eta = 0.3 from (w0, c0), then a and b (equation 3.8)
a_std = None
b_std = None

In [ ]:
# Check: equation 3.9 and np.polyfit
a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'equation 3.9: a = {a_ls:.3f}, b = {b_ls:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Complete the previous cell.')
else:
    print('line obtained:', 'correct' if np.allclose([a_std, b_std], [a_ls, b_ls], atol=1e-4) else 'check again')

### e) and f) The loss of the two descents, then the number of steps

In [ ]:
# The two descents of lesson 3.5, step by step (nothing to complete)
def path_m2(n_steps, eta=0.00006):
    a, b, losses = 2.5, 30.0, []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return losses

def path_std(n_steps, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, losses = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return losses

plt.figure(figsize=(6, 3))
plt.semilogy(path_m2(30), marker='o', color=ORANGE, label='areas in m², η = 0.00006')
plt.semilogy(path_std(30), marker='s', color=BLUE, label='standardized areas, η = 0.3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Number of steps'); plt.ylabel('Loss (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Number of steps to get within 0.1 k€² of the minimum (a few seconds)
L_min = np.mean((y - (a_ls * x + b_ls)) ** 2)

def n_steps(X, a, b, eta, limit=10 ** 6):
    for k in range(1, limit + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

losses_std = path_std(30)
print('standardized areas, η = 0.3:', next(k for k, v in enumerate(losses_std) if v - L_min < 0.1), 'steps')
print('areas in m², η = 0.00006:      ', n_steps(x, 2.5, 30.0, 0.00006), 'steps')

*Your interpretation:* …

## Exercise 3.5 — Predict, then check: areas in tens of m²

*Data D0.1. Concepts: change of scale (lesson 2.5), gradient (lesson 3.4), standardization (lesson 3.5).*

### a) to d) Your predictions, before running

*a):* …

*b):* …

*c):* …

*d):* …

### e) Check

In [ ]:
# TO COMPLETE: the areas of A to F in tens of m²
x_tens = None

In [ ]:
# Check, then the measurements (nothing else to complete)
if x_tens is None:
    print('Complete the previous cell.')
else:
    print('x_tens:', 'correct' if np.allclose(x_tens, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'check again')
    print('a) least-squares line (slope, intercept):', np.round(np.polyfit(x_tens, y, 1), 2))
    e0 = y - (25 * x_tens + 30)
    print(f'b) quotient at step 0: {np.sum(x_tens * e0) / np.sum(e0):.2f} tens of m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_tens - b
            a, b = a + eta * 2 / 6 * np.sum(x_tens * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: loss after 200 steps = {np.mean((y - a * x_tens - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_tens - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_tens * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_tens - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0.006: {k} steps to get within 0.1 k€² of the minimum')

### f) Interpret

*Your answer:* …

## Exercise 3.6 — Challenge: circles on the map

*Data D0.1. Concepts: standardization (lessons 1.4 and 3.5), partial derivatives and gradient (lesson 3.4).* Questions a) to f) are done on paper. The notebook checks your results.

### a) to f) On paper

*Your proofs:* …

### Check in the notebook

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) sum of the z = {np.sum(zz):.1e}, sum of the z² = {np.sum(zz ** 2):.4f}')

In [ ]:
# TO COMPLETE: m = (1/n) Σ z_i y_i, then one descent step with learning rate eta from (w, c)
m = None

def step(w, c, eta):
    """Returns (w, c) after one update (equation 3.7 with z instead of x)."""
    return None   # TO COMPLETE

In [ ]:
# Check of b) and d)
if m is None or step(1, 1, 0.1) is None:
    print('Complete the previous cell.')
else:
    print(f'b) m = {m:.2f} →', 'correct' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'check again')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = step(w0, c0, 0.5)
    print(f'd) one step with η = 0.5: (w, c) = ({w1:.2f}, {c1:.2f}) →',
          'correct' if np.allclose([w1, c1], [m, y.mean()]) else 'check again')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        sequence = []
        for _ in range(4):
            w, c = step(w, c, eta)
            sequence.append(round(c, 2))
        print(f'   η = {eta}: c after 1, 2, 3, 4 steps = {sequence}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLUE)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ per standard deviation)'); plt.ylabel('c (k€)'); plt.title('e) Circles'); plt.show()